**Importer les libraries pertinentes**

Source : Prettenhofer, P., Grisel, O., Blondel, M., Amor, A. et Buitinck, L. [Classification of text documents using sparse features](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py). *scikit-learn*. 

In [10]:
import multiprocessing
import pandas as pd
from os import listdir
from os import path

from IPython.display import clear_output

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes
from sklearn.naive_bayes import MultinomialNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn import svm

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouche
#from sklearn.neural_network import MLPClassifier

**Charger les données**

In [11]:
folder = '../1-data/training_datasets/'
datasets = listdir(folder)
datasets = datasets[3:4]
datasets

['train_dataset_40pc.xlsx']

**Entraîner les modèles et sortir les résultats**

In [12]:
report = []
for file in datasets:
    ratio = file[14:-7]
    
    df = pd.read_excel(path.join(folder, file))
    X = df['text_post'].astype(str)
    y = df['category']

    # 5-fold cross-validation
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 1000, 2500, 5000, 10000, 15000]

    # Antidictionnaire personnalisé
    with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
        custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
        custom_stopwords += list(ENGLISH_STOP_WORDS)

    # Algorithmes à tester
    algorithms = {
        'Naive Bayes' : MultinomialNB(), 
        'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        'Support Vector Machines (SVM)' :  svm.SVC(kernel="linear", C=1.0),
        'Logistic Regression': LogisticRegression(C=5, max_iter=1000),
        'Random Forest': RandomForestClassifier(),
        #'Multilayered Perceptron': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
    }

    for n_features in n_features_values:
            tfidf_bow_vectorizer = TfidfVectorizer(stop_words=custom_stopwords, max_features=n_features)
            
            for algorithm in algorithms.keys():
                pipeline_algo = Pipeline([
                    ('tfidf', tfidf_bow_vectorizer),
                    ('classifier', algorithms[algorithm])
                ])

                # Validation croisée 
                scores = cross_val_score(pipeline_algo, X, y, cv=stratified_kfold, scoring='accuracy')
                predictions = cross_val_predict(pipeline_algo, X, y, cv=stratified_kfold)

                # Performances (rappel, précision, score F)
                results = {
                    '% Incels' : int(ratio),
                    'Algorithme' : algorithm,
                    'N traits discr.' : n_features,
                    'accuracy': scores.mean()       
                }
                results.update(classification_report(y, predictions, output_dict=True)['macro avg'])
                report.append(results)
                
                print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
                clear_output(wait=True)

Random Forest  | 40% Incels |  15000 traits discr.
               precision    recall  f1-score   support

       incel       0.73      0.62      0.67     20000
     neutral       0.77      0.85      0.81     30000

    accuracy                           0.76     50000
   macro avg       0.75      0.74      0.74     50000
weighted avg       0.76      0.76      0.75     50000



**Résultats**

In [13]:
# N.B. : Support = Nombre d'occurrence dans chaque classe
report = pd.DataFrame(report)
report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

report = report.rename(columns={'support':'nb_posts_total'})
report['nb_posts_total'] = report['nb_posts_total'].astype(int)

# Réorganiser l'ordre des colonnes
report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

report

,nb_posts_total,nb_posts_incels,nb_posts_neutral,% Incels,Algorithme,N traits discr.,accuracy,precision,recall,f1-score
0,50000,20000,30000,40,Naive Bayes,100,0.68184,0.738105,0.612383,0.591750
1,50000,20000,30000,40,kNN (k=1),100,0.42050,0.487162,0.493875,0.375621
2,50000,20000,30000,40,kNN (k=2),100,0.41914,0.514399,0.504192,0.350814
3,50000,20000,30000,40,kNN (k=3),100,0.42890,0.502151,0.501075,0.387269
4,50000,20000,30000,40,Support Vector Machines (SVM),100,0.69142,0.719566,0.631475,0.622996
5,50000,20000,30000,40,Logistic Regression,100,0.69482,0.710437,0.640533,0.636849
6,50000,20000,30000,40,Random Forest,100,0.68768,0.691514,0.636825,0.634564
7,50000,20000,30000,40,Naive Bayes,250,0.71126,0.748175,0.653750,0.650023
8,50000,20000,30000,40,kNN (k=1),250,0.45066,0.517895,0.511783,0.426986
9,50000,20000,30000,40,kNN (k=2),250,0.43550,0.542589,0.516042,0.379875


**Exporter les résultats de l'apprentissage**

In [14]:
report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_40pc.csv', index=False
)

**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour le système retenu uniquement*

In [15]:
file_train = '../1-data/training_datasets/train_dataset_40pc.xlsx'

df_train = pd.read_excel(file_train)
df_train

,id_post,date_post,subreddit,author,text_post,category
0,i1ihqm5,2022,ForeverAlone,borgwardB,find a hobby you're interested in that has a l...,incel
1,hklyo5r,2021,ForeverAlone,Polaris022,"Well, you can be realistic about your looks an...",incel
2,dnr6u54,2017,ForeverAlone,changeIsTheWay,Oh yeah. You're the chick who I commented on a...,incel
3,dldhg5i,2017,Incels,Pr_Revolving,Honestly this is beyond disturbing.,incel
4,g3ka3n0,2020,IncelsWithoutHate,Intelligent_Badger_1,Is it really true that americans dislike sharp...,incel
...,...,...,...,...,...,...
49995,esg0s29,2019,financialindependence,sulli_p,"Okay I’m fairly new to investing, what about s...",neutral
49996,fzxnufo,2020,realtors,VaBeachRealtor,How is it not? \n\nJust because it comes with ...,neutral
49997,hv2cgip,2022,exjw,DuchessBolo,Absolutely!,neutral
49998,hv2cvjs,2022,tuckedinfishies,thousandlotuspetals_,Mine does this all the time On her piece of wo...,neutral


In [16]:
# Système retenu : Naive Bayes ; 40% de données Incels ; 5 000 traits discriminants
X_train = df_train['text_post'].astype(str)
y_train = df_train['category']

ngram_values = (1, 2)

# Antidictionnaire personnalisé
with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
    custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
    custom_stopwords += list(ENGLISH_STOP_WORDS)

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=15000)),
    ('nb', MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=15000, ngram_range=(1, 2),
                                 stop_words=['\ufeff&gt', 'a', 'â', 'ä', 'ã',
                                             'å', 'ªã', 'about', 'actually',
                                             'æ', 'again', 'against', "ain't",
                                             'all', 'allow', 'allows', 'almost',
                                             'alone', 'along', 'already',
                                             'also', 'although', 'always', 'am',
                                             'among', 'amongst', 'amp', 'an',
                                             'and', 'another', ...])),
                ('nb', MultinomialNB())])

In [17]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_predictions = cross_val_predict(pipeline_nb, X_train, y_train, cv=stratified_kfold)

# Performances en phase d'apprentissage du modèle retenu (rappel, précision, score F)
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_train, nb_predictions)
precision_nb = precision_score(y_train, nb_predictions, average='macro')
recall_nb = recall_score(y_train, nb_predictions, average='macro')
f1_nb = f1_score(y_train, nb_predictions, average='macro')

df = [
    {
        'Phase': 'Apprentissage',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
]

print(classification_report(y_train, nb_predictions))

              precision    recall  f1-score   support

       incel       0.80      0.62      0.70     20000
     neutral       0.78      0.89      0.83     30000

    accuracy                           0.78     50000
   macro avg       0.79      0.76      0.77     50000
weighted avg       0.79      0.78      0.78     50000



*On teste sur de nouvelles données*

In [18]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

,date_post,id_post,author,subreddit,text_post,category
0,2021,hewyqir,KallystoNyx,FreeKarma4U,Bang!,neutral
1,2022,hv2cm15,Hopeful-Talk-1556,canada,I mean this is why you often get downvoted in ...,neutral
2,2021,hewyfkk,naughtyrpfan,HentaiAndRoleplayy,You playing m or f?,neutral
3,2016,d1l6201,greigmusic,IAmA,"Lol, google black pudding and your view will c...",neutral
4,2020,fzxnd3h,NorCalifornioAH,MapPorn,That's more a matter of Combined Statistical A...,neutral
...,...,...,...,...,...,...
9995,2021,hgdqoim,throwamay555,ForeverAlone,who said you were ugly?,incel
9996,2021,hmqkhx0,Admirable_Elk_965,ForeverAlone,Yeah I see this too when I walk out of class. ...,incel
9997,2022,iw0zptn,Resident_Setting3884,ForeverAlone,"this is exactly how i feel, im a cab driver, t...",incel
9998,2017,dfiq9mw,CALLMEMANTEE,Incels,I mean I'm not ok with it because its depressi...,incel


*Vérifier qu'aucune donnée test n'était contenue dans les données d'appentissage*

In [19]:
validate = pd.concat([df_train, df_test])

validate[validate.duplicated(subset='id_post')]

,id_post,date_post,subreddit,author,text_post,category


*Appliquer le classifieur aux nouvelles données*

In [20]:
X_test = df_test['text_post'].astype(str)
y_test = df_test['category']

y_pred_nb = pipeline_nb.predict(X_test)

In [21]:
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')

df.append(
    {
        'Phase': 'Test',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
)

pd.set_option("display.precision", 4)
df = pd.DataFrame(df)
df.to_csv('../3-results/results_test.csv', index=False)
df

,Phase,Précision,Rappel,Mesure F,Exactitude
0,Apprentissage,0.7878,0.7573,0.7651,0.7846
1,Test,0.6753,0.7566,0.7033,0.8675


In [22]:
accuracy_nb

0.8675

In [23]:
precision_nb

0.6753170186405374

In [24]:
recall_nb

0.7566111111111111

In [25]:
f1_nb

0.7033224238831136

In [26]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

       incel       0.40      0.62      0.48      1000
     neutral       0.95      0.90      0.92      9000

    accuracy                           0.87     10000
   macro avg       0.68      0.76      0.70     10000
weighted avg       0.90      0.87      0.88     10000



In [27]:
import pandas as pd
results = pd.read_csv('../3-results/results_training.csv')
results.sort_values(by='f_score', ascending=False)

KeyError: 'f_score'